<a href="https://www.kaggle.com/code/aabdollahii/knn-iris?scriptVersionId=372299256" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

# Load Iris dataset
iris = load_iris()

X = iris.data
y = iris.target

feature_names = iris.feature_names
class_names = iris.target_names

print("Data shape:", X.shape)
print("Labels shape:", y.shape)
print("Features:", feature_names)
print("Classes:", class_names)

Data shape: (150, 4)
Labels shape: (150,)
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: ['setosa' 'versicolor' 'virginica']


## data analysis 

In [3]:
df = pd.DataFrame(X, columns=feature_names)
df["class"] = y

print(df.head())
print("\nMissing values:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df["class"].value_counts().sort_index())

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   class  
0      0  
1      0  
2      0  
3      0  
4      0  

Missing values:
sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
class                0
dtype: int64

Class distribution:
class
0    50
1    50
2    50
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (120, 4)
Test: (30, 4)


## Standardization

In [5]:
# Calculate statistics using training data only
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# Standardization
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

These lines of code help to standardize the output in later works and ensure fairness.


## writing KNN only for one feature

In [6]:


def calculate_distances(x_value, feature_train):

    distances = np.abs(feature_train - x_value)

    return distances



# testing calculate distance 
x_value = 4.8

feature_train = np.array([1.4, 4.5, 4.9, 5.1, 4.7])
distances = calculate_distances(x_value, feature_train)

print(distances)

[3.4 0.3 0.1 0.3 0.1]


In [7]:

def get_neighbors(distances, y_train, k):

    sorted_indices = np.argsort(distances)

    neighbor_indices = sorted_indices[:k]

    neighbor_labels = y_train[neighbor_indices]

    return neighbor_indices, neighbor_labels

# We need to use the sort function to assign the closest dots to the correct class. The whole function: these lines  of code are sorting

In [8]:

def knn_predict(x_value, feature_train, y_train, k):

    k = max(1, min(k, len(y_train)))

    distances = calculate_distances(x_value, feature_train)

    neighbor_indices, neighbor_labels = get_neighbors(
        distances, y_train, k
    )

    class_counts = np.bincount(neighbor_labels, minlength=3)

    max_count = np.max(class_counts)

    predicted_class = neighbor_labels[0]

    for i in range(len(neighbor_labels)):

        current_class = neighbor_labels[i]

        if class_counts[current_class] == max_count:

            predicted_class = current_class

            break

    return predicted_class, class_counts


### testing KNN

In [9]:
feature_train = np.array([1.4, 4.5, 4.9, 5.1, 4.7])

y_train = np.array([0, 1, 2, 1, 2])

x_value = 4.8

k = 5

distances = calculate_distances(x_value, feature_train)

neighbor_indices, neighbor_labels = get_neighbors(
    distances, y_train, k
)

predicted_class, class_counts = knn_predict(
    x_value, feature_train, y_train, k
)

print("Distances:", distances)

print("Neighbor indices:", neighbor_indices)

print("Neighbor labels:", neighbor_labels)

print("Class counts:", class_counts)

print("Predicted class:", predicted_class)

Distances: [3.4 0.3 0.1 0.3 0.1]
Neighbor indices: [4 2 3 1 0]
Neighbor labels: [2 2 1 1 0]
Class counts: [1 2 2]
Predicted class: 2


In [10]:
# We had an error using KNN on Iris, then we used 
from sklearn.model_selection import train_test_split

_, _, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(len(y_train))
print(len(y_test))

120
30


In [11]:
# test KNN on IRIS
k = 5

sample_index = 0
feature_index = 2

x_value = X_test[sample_index, feature_index]

feature_train = X_train[:, feature_index]

predicted_class, class_counts = knn_predict(
    x_value,
    feature_train,
    y_train,
    k
)

print("Test value:", x_value)
print("Class counts:", class_counts)
print("Predicted class:", class_names[predicted_class])
print("Actual class:", class_names[y_test[sample_index]])


Test value: -1.402503840804636
Class counts: [5 0 0]
Predicted class: setosa
Actual class: setosa


In [12]:

k = 5

feature_index = 2

feature_train = X_train[:, feature_index]

predictions = np.zeros(len(X_test), dtype=int)

for i in range(len(X_test)):

    x_value = X_test[i, feature_index]

    predicted_class, class_counts = knn_predict(
        x_value,
        feature_train,
        y_train,
        k
    )

    predictions[i] = predicted_class


correct = predictions == y_test

accuracy = np.mean(correct)

print("Correct predictions:", np.sum(correct))

print("Total predictions:", len(y_test))

print("Accuracy:", accuracy)
print("Accuracy percentage:", accuracy * 100)


Correct predictions: 28
Total predictions: 30
Accuracy: 0.9333333333333333
Accuracy percentage: 93.33333333333333


In [13]:
# run all of them using different KNN 






k = 5

n_test = len(X_test)
n_features = X_train.shape[1]

all_predictions = np.zeros((n_test, n_features), dtype=int)

feature_accuracies = np.zeros(n_features)

for j in range(n_features):

    feature_train = X_train[:, j]

    for i in range(n_test):

        x_value = X_test[i, j]

        predicted_class, class_counts = knn_predict(
            x_value,
            feature_train,
            y_train,
            k
        )

        all_predictions[i, j] = predicted_class

    correct = all_predictions[:, j] == y_test

    accuracy = np.mean(correct)

    feature_accuracies[j] = accuracy

    print("Feature:", feature_names[j])
    print("Correct:", np.sum(correct))
    print("Accuracy:", accuracy * 100)


Feature: sepal length (cm)
Correct: 17
Accuracy: 56.666666666666664
Feature: sepal width (cm)
Correct: 18
Accuracy: 60.0
Feature: petal length (cm)
Correct: 28
Accuracy: 93.33333333333333
Feature: petal width (cm)
Correct: 28
Accuracy: 93.33333333333333


# Parzen Part 

$$
\hat{f}(x) = \frac{1}{n h} \sum_{i=1}^{n} K\left(\frac{x - x_i}{h}\right), \quad \text{where } K(z) = \frac{1}{\sqrt{2\pi}} e^{-\frac{1}{2}z^2}
$$

In [14]:

def parzen_density(x_value, feature_class, h):

    n = len(feature_class)

    z = (x_value - feature_class) / h
    
    kernel_values = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * z**2)

    density = np.sum(kernel_values) / (n * h)

    return density


In [15]:
#initial test



h = 0.5

sample_index = 0
feature_index = 2

x_value = X_test[sample_index, feature_index]

density_scores = np.zeros(3)

for class_id in range(3):

    feature_class = X_train[y_train == class_id, feature_index]

    density_scores[class_id] = parzen_density(
        x_value,
        feature_class,
        h
    )

    print("Class:", class_names[class_id])

    print("Sample count:", len(feature_class))

    print("Density:", density_scores[class_id])



predicted_class = np.argmax(density_scores)

print("Predicted class:", class_names[predicted_class])

print("Actual class:", class_names[y_test[sample_index]])


Class: setosa
Sample count: 40
Density: 0.7702028826459716
Class: versicolor
Sample count: 40
Density: 0.008977086924467016
Class: virginica
Sample count: 40
Density: 6.787364526263794e-05
Predicted class: setosa
Actual class: setosa


In [16]:

h = 0.5

n_test = len(X_test)
n_features = X_train.shape[1]
n_classes = len(class_names)

parzen_predictions = np.zeros(
    (n_test, n_features), dtype=int
)

parzen_density_scores = np.zeros(
    (n_test, n_features, n_classes)
)

for i in range(n_test):

    for j in range(n_features):

        x_value = X_test[i, j]

        for class_id in range(n_classes):

            feature_class = X_train[
                y_train == class_id, j
            ]

            density = parzen_density(
                x_value,
                feature_class,
                h
            )

            parzen_density_scores[i, j, class_id] = density

        predicted_class = np.argmax(
            parzen_density_scores[i, j, :]
        )

        parzen_predictions[i, j] = predicted_class


## four different parzen  

In [17]:

parzen_accuracies = np.zeros(n_features)

for j in range(n_features):

    correct = parzen_predictions[:, j] == y_test

    accuracy = np.mean(correct)

    parzen_accuracies[j] = accuracy

    print("Feature:", feature_names[j])

    print("Correct:", np.sum(correct))

    print("Accuracy:", accuracy * 100)



Feature: sepal length (cm)
Correct: 17
Accuracy: 56.666666666666664
Feature: sepal width (cm)
Correct: 21
Accuracy: 70.0
Feature: petal length (cm)
Correct: 27
Accuracy: 90.0
Feature: petal width (cm)
Correct: 28
Accuracy: 93.33333333333333


# Voting sys

In [18]:

n_test = len(X_test)
n_classes = len(class_names)
n_features = X_train.shape[1]

combined_scores = np.zeros((n_test, n_classes))

weight = 0.125

for i in range(n_test):

    for j in range(n_features):

        knn_class = all_predictions[i, j]

        parzen_class = parzen_predictions[i, j]

        combined_scores[i, knn_class] += weight

        combined_scores[i, parzen_class] += weight


combined_predictions = np.argmax(combined_scores, axis=1)


## eval voting system

In [19]:

correct = combined_predictions == y_test

combined_accuracy = np.mean(correct)

print("Correct predictions:", np.sum(correct))

print("Total predictions:", len(y_test))

print("Combined Accuracy:", combined_accuracy * 100)


Correct predictions: 27
Total predictions: 30
Combined Accuracy: 90.0


In [20]:
combined_results = pd.DataFrame(
    combined_scores,
    columns=class_names
)

combined_results.insert(
    0, "Actual", class_names[y_test]
)

combined_results.insert(
    1, "Predicted", class_names[combined_predictions]
)

print(combined_results.head(10))

       Actual   Predicted  setosa  versicolor  virginica
0      setosa      setosa   0.750       0.000      0.250
1   virginica   virginica   0.000       0.375      0.625
2  versicolor  versicolor   0.250       0.625      0.125
3  versicolor  versicolor   0.250       0.750      0.000
4      setosa      setosa   1.000       0.000      0.000
5  versicolor  versicolor   0.125       0.500      0.375
6      setosa      setosa   1.000       0.000      0.000
7      setosa      setosa   1.000       0.000      0.000
8   virginica   virginica   0.000       0.000      1.000
9  versicolor  versicolor   0.250       0.500      0.250


In [21]:
print(combined_scores.sum(axis=1))

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1.]


# experiment different H and K parameter(we can call it parameter tunning maybe)

In [22]:

from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train_raw, y_test_final = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_subtrain_raw, X_val_raw, y_subtrain, y_val = train_test_split(
    X_train_raw,
    y_train_raw,
    test_size=0.25,
    random_state=21,
    stratify=y_train_raw
)

mean_exp = X_subtrain_raw.mean(axis=0)
std_exp = X_subtrain_raw.std(axis=0)

X_subtrain = (X_subtrain_raw - mean_exp) / std_exp
X_val = (X_val_raw - mean_exp) / std_exp

print("Training:", X_subtrain.shape)
print("Validation:", X_val.shape)
print("Final Test:", X_test_raw.shape)


Training: (90, 4)
Validation: (30, 4)
Final Test: (30, 4)


In [23]:

k_values = np.array([1, 3, 5, 7, 9, 11])

h_values = np.array([0.1, 0.2, 0.3, 0.5, 0.7, 1.0])

results = np.zeros((len(k_values), len(h_values)))

print("K values:", k_values)
print("h values:", h_values)
print("Results shape:", results.shape)


K values: [ 1  3  5  7  9 11]
h values: [0.1 0.2 0.3 0.5 0.7 1. ]
Results shape: (6, 6)


In [24]:

n_val = len(X_val)
n_features = X_val.shape[1]
n_classes = len(class_names)

for k_index in range(len(k_values)):

    k = k_values[k_index]

    knn_predictions_val = np.zeros(
        (n_val, n_features), dtype=int
    )

    for j in range(n_features):

        feature_train = X_subtrain[:, j]

        for i in range(n_val):

            x_value = X_val[i, j]

            predicted_class, class_counts = knn_predict(
                x_value,
                feature_train,
                y_subtrain,
                k
            )

            knn_predictions_val[i, j] = predicted_class

    for h_index in range(len(h_values)):

        h = h_values[h_index]

        parzen_predictions_val = np.zeros(
            (n_val, n_features), dtype=int
        )

        for i in range(n_val):

            for j in range(n_features):

                x_value = X_val[i, j]

                density_scores = np.zeros(n_classes)

                for class_id in range(n_classes):

                    feature_class = X_subtrain[
                        y_subtrain == class_id, j
                    ]

                    density_scores[class_id] = parzen_density(
                        x_value,
                        feature_class,
                        h
                    )

                predicted_class = np.argmax(density_scores)

                parzen_predictions_val[i, j] = predicted_class

        combined_scores_val = np.zeros(
            (n_val, n_classes)
        )

        weight = 0.125

        for i in range(n_val):

            for j in range(n_features):

                knn_class = knn_predictions_val[i, j]

                parzen_class = parzen_predictions_val[i, j]

                combined_scores_val[i, knn_class] += weight

                combined_scores_val[i, parzen_class] += weight

        combined_predictions_val = np.argmax(
            combined_scores_val, axis=1
        )

        accuracy = np.mean(
            combined_predictions_val == y_val
        )

        results[k_index, h_index] = accuracy

        print(
            "K:", k,
            "h:", h,
            "Accuracy:", accuracy * 100
        )


K: 1 h: 0.1 Accuracy: 93.33333333333333
K: 1 h: 0.2 Accuracy: 93.33333333333333
K: 1 h: 0.3 Accuracy: 93.33333333333333
K: 1 h: 0.5 Accuracy: 86.66666666666667
K: 1 h: 0.7 Accuracy: 86.66666666666667
K: 1 h: 1.0 Accuracy: 86.66666666666667
K: 3 h: 0.1 Accuracy: 93.33333333333333
K: 3 h: 0.2 Accuracy: 93.33333333333333
K: 3 h: 0.3 Accuracy: 93.33333333333333
K: 3 h: 0.5 Accuracy: 90.0
K: 3 h: 0.7 Accuracy: 90.0
K: 3 h: 1.0 Accuracy: 90.0
K: 5 h: 0.1 Accuracy: 90.0
K: 5 h: 0.2 Accuracy: 90.0
K: 5 h: 0.3 Accuracy: 90.0
K: 5 h: 0.5 Accuracy: 86.66666666666667
K: 5 h: 0.7 Accuracy: 86.66666666666667
K: 5 h: 1.0 Accuracy: 86.66666666666667
K: 7 h: 0.1 Accuracy: 93.33333333333333
K: 7 h: 0.2 Accuracy: 93.33333333333333
K: 7 h: 0.3 Accuracy: 93.33333333333333
K: 7 h: 0.5 Accuracy: 93.33333333333333
K: 7 h: 0.7 Accuracy: 93.33333333333333
K: 7 h: 1.0 Accuracy: 93.33333333333333
K: 9 h: 0.1 Accuracy: 93.33333333333333
K: 9 h: 0.2 Accuracy: 93.33333333333333
K: 9 h: 0.3 Accuracy: 93.3333333333333

In [25]:

results_df = pd.DataFrame(
    results,
    index=k_values,
    columns=h_values
)

results_df.index.name = "K"
results_df.columns.name = "h"

print((results_df * 100).round(2))


h     0.1    0.2    0.3    0.5    0.7    1.0
K                                           
1   93.33  93.33  93.33  86.67  86.67  86.67
3   93.33  93.33  93.33  90.00  90.00  90.00
5   90.00  90.00  90.00  86.67  86.67  86.67
7   93.33  93.33  93.33  93.33  93.33  93.33
9   93.33  93.33  93.33  93.33  93.33  93.33
11  90.00  90.00  90.00  86.67  86.67  86.67


In [26]:

best_index = np.unravel_index(
    np.argmax(results),
    results.shape
)

best_k = k_values[best_index[0]]

best_h = h_values[best_index[1]]

best_accuracy = results[best_index]

print("Best K:", best_k)

print("Best h:", best_h)

print("Best Accuracy:", best_accuracy * 100)


Best K: 1
Best h: 0.1
Best Accuracy: 93.33333333333333


# eval with best parameter

In [27]:

best_k = 1
best_h = 0.1

n_test = len(X_test)
n_features = X_train.shape[1]
n_classes = len(class_names)

final_knn_predictions = np.zeros(
    (n_test, n_features), dtype=int
)

final_parzen_predictions = np.zeros(
    (n_test, n_features), dtype=int
)

final_scores = np.zeros((n_test, n_classes))

for i in range(n_test):

    for j in range(n_features):

        feature_train = X_train[:, j]

        x_value = X_test[i, j]

        knn_class, class_counts = knn_predict(
            x_value,
            feature_train,
            y_train,
            best_k
        )

        final_knn_predictions[i, j] = knn_class

        density_scores = np.zeros(n_classes)

        for class_id in range(n_classes):

            feature_class = X_train[
                y_train == class_id, j
            ]

            density_scores[class_id] = parzen_density(
                x_value,
                feature_class,
                best_h
            )

        parzen_class = np.argmax(density_scores)

        final_parzen_predictions[i, j] = parzen_class

        final_scores[i, knn_class] += 0.125

        final_scores[i, parzen_class] += 0.125


final_predictions = np.argmax(final_scores, axis=1)

correct = final_predictions == y_test

final_accuracy = np.mean(correct)

print("Best K:", best_k)
print("Best h:", best_h)

print("Correct predictions:", np.sum(correct))

print("Total predictions:", len(y_test))

print("Final Test Accuracy:", final_accuracy * 100)


Best K: 1
Best h: 0.1
Correct predictions: 24
Total predictions: 30
Final Test Accuracy: 80.0


In [28]:

final_confusion = np.zeros((3, 3), dtype=int)

for i in range(len(y_test)):

    actual = y_test[i]

    predicted = final_predictions[i]

    final_confusion[actual, predicted] += 1


final_confusion_df = pd.DataFrame(
    final_confusion,
    index=class_names,
    columns=class_names
)

print(final_confusion_df)


            setosa  versicolor  virginica
setosa          10           0          0
versicolor       0           6          4
virginica        0           2          8


In [29]:

knn_final_accuracy = np.zeros(n_features)
parzen_final_accuracy = np.zeros(n_features)

for j in range(n_features):

    knn_correct = final_knn_predictions[:, j] == y_test

    parzen_correct = final_parzen_predictions[:, j] == y_test

    knn_final_accuracy[j] = np.mean(knn_correct)

    parzen_final_accuracy[j] = np.mean(parzen_correct)


final_comparison = pd.DataFrame({
    "Feature": feature_names,
    "KNN Accuracy": knn_final_accuracy * 100,
    "Parzen Accuracy": parzen_final_accuracy * 100
})

print(final_comparison.round(2))


             Feature  KNN Accuracy  Parzen Accuracy
0  sepal length (cm)         50.00            56.67
1   sepal width (cm)         40.00            56.67
2  petal length (cm)         86.67            93.33
3   petal width (cm)         83.33            93.33
